# QLoRA SFT：Gemma 4 12B → PCB placement (genpcb)

在 **Colab Pro+** 微調（A100 40GB 最佳；L4 24GB 4-bit 也可）。

- **任務**：placement —— prompt（板框+元件宣告+netlist）→ completion（擺位 P 行），completion-only loss
- **資料**：本 notebook 直接用 repo 的程序化生成器產出（不需上傳）
- **流程**：裝套件 → clone repo → 產資料 → 4-bit 載入 → QLoRA SFT → 存 adapter → 推論並用 reward 檢查

> 執行前：Runtime → Change runtime type → GPU(A100 / L4)。
> ⚠️ Gemma 是 **gated**：先用你的 HF 帳號到 huggingface.co/google/gemma-4-12b-it 按 Accept。
> Colab Secrets(🔑) 需設 `HF_TOKEN`、`GH_TOKEN`（clone private repo）。

## 1. 確認 GPU

In [ ]:
!nvidia-smi

## 2. 安裝套件
若日後 TRL/transformers API 變動，主要調整點是 `SFTConfig` 參數名（`completion_only_loss` / `max_length`）與 `SFTTrainer` 的 `processing_class`/`tokenizer`。

In [ ]:
!pip -q install -U \
  "transformers>=5.0" "trl>=0.15" "peft>=0.13" \
  "bitsandbytes>=0.44" "accelerate>=1.0" "datasets>=3.0"

## 3. 設定（可直接改這格）

In [ ]:
MODEL      = "google/gemma-4-12b-it"   # A/B：改 "Qwen/Qwen3.5-9B" 即切換；GPU 弱可用較小杯
N_BOARDS   = 3000                       # 程序化板數
GRID_MM    = 0.1
VAL_FRAC   = 0.1                        # 以 netlist 為單位切（防洩漏）
SEED       = 17
MAX_SEQ_LEN = 2048                      # 板 DSL 多 <1.1k token，2048 充裕
EPOCHS     = 2
DATA_PATH  = "/content/drive/MyDrive/genpcb/data/sft/boards.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/genpcb/qlora-gemma4-12b"

## 4. Secrets + 掛 Drive + 取得 repo 邏輯
資料生成 / DSL 切分 / reward 都重用 repo 內已測程式碼，不複製進 notebook。

In [ ]:
import os, sys
from google.colab import drive, userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")        # Gemma 下載（須先在 HF 接受授權）
drive.mount("/content/drive")
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

GH = userdata.get("GH_TOKEN")
if os.path.isdir("/content/genpcb"):
    !git -C /content/genpcb pull
else:
    !git clone https://{GH}@github.com/timmytsaa/genpcb.git /content/genpcb
!pip -q install -e /content/genpcb
sys.path.insert(0, "/content/genpcb/src")

## 5. 產生訓練資料（程序化，呼叫 repo）
prompt/completion 為 placement 任務；以 netlist 為單位切 train/val 防洩漏。

In [ ]:
from datasets import Dataset
from genpcb.data.build import build
from genpcb.train.sft import load_examples, split_by_netlist

build(N_BOARDS, DATA_PATH, grid=GRID_MM)                 # → boards.jsonl（存 Drive）
examples = load_examples(DATA_PATH)
train, val = split_by_netlist(examples, VAL_FRAC, SEED)
to_ds = lambda rows: Dataset.from_list(
    [{"prompt": e["prompt"], "completion": e["completion"]} for e in rows])
train_ds, val_ds = to_ds(train), to_ds(val)
print(f"{len(train_ds)} train / {len(val_ds)} val")
print("--- 範例 prompt ---\n" + train_ds[0]["prompt"][:300])
print("--- 範例 completion ---\n" + train_ds[0]["completion"][:200])

## 6. 4-bit 載入模型 + tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)
model.config.use_cache = False   # 與 gradient checkpointing 相容

## 7. LoRA 設定

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
peft_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

## 8. 訓練（QLoRA SFT，completion-only）
用 prompt/completion 資料集 + `completion_only_loss=True`，只在擺位上算 loss。
**不套 chat template**：與 GRPO 階段餵的原始 prompt 一致，避免 train/RL 不匹配。

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=4,      # OOM 就降 2 → 1
    gradient_accumulation_steps=4,      # 有效 batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    max_length=MAX_SEQ_LEN,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
    processing_class=tok,               # 舊版 TRL 用 tokenizer=tok
)
trainer.train()

## 9. 存 LoRA adapter

In [ ]:
trainer.save_model(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)
print("adapter 已存到:", OUTPUT_DIR)

## 10. 推論 + reward 檢查
給 held-out netlist 的 prompt，生成擺位，用 `reward_from_completion` 評分：
reward > -100 代表輸出**良構且完整**（能 parse 成合法擺位）；數值越高擺位越好。
比印出文字用眼睛看更直接——這就是 SFT 煙霧測試要的訊號。

In [ ]:
from genpcb.rewards import reward_from_completion

model.config.use_cache = True
prompt = val_ds[0]["prompt"]
inputs = tok(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
gen = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== prompt ===\n" + prompt)
print("=== generated placement ===\n" + gen)
r_gen = reward_from_completion(prompt, gen)
r_ref = reward_from_completion(prompt, val_ds[0]["completion"])
print(f"\nreward(generated) = {r_gen:.3f}   |   reward(ground-truth) = {r_ref:.3f}")
print("良構且完整：" + ("是" if r_gen > -100 else "否（仍在學格式）"))

## 調參小抄 / 下一步

- **OOM**：`per_device_train_batch_size` 4→2→1、或降 `MAX_SEQ_LEN`、或換較小模型
- **TRL 報參數錯**：把錯誤訊息對照 `SFTConfig` 簽名調（`completion_only_loss`/`max_length`/`eval_strategy`）
- **reward 一直 = -100**：模型還沒學會輸出完整 DSL → 加 epoch 或加 `N_BOARDS`
- **過擬合**（val loss 上升）：降 epoch

**煙霧測試成功標準**：train/val loss 下降 + 推論 reward 脫離 -100（會 parse）。
通過後 → 接 GRPO（`reward_fn` 已就緒，見 src/genpcb/train/grpo.py）。